# Wakil AI — Phase 3 (Part 2): A Transformer From Scratch

The **Transformer** (Vaswani et al., 2017, *"Attention Is All You Need"*) is the architecture behind GPT, Claude, Llama and Gemini. In this notebook you'll see every piece of a small one, **written in NumPy with backpropagation by hand**:

1. **Self-attention**: computed by hand on a tiny example, then the real code
2. **Positional embeddings**, **LayerNorm**, **residual connections**: why each exists
3. **Adam**: the optimizer used to train almost every Transformer
4. Training on Shakespeare, a comparison with the Part 1 model, and a **look inside the attention heads**

| | |
|---|---|
| ✅ Implemented from scratch | NumPy only, forward **and** backward (`wakil_ai/transformer.py`) |
| ✅ Trained from scratch | Random initial weights, learns only from Tiny Shakespeare |
| ❌ Pretrained weights / fine-tuning / AI API | Not used |

**Runtime → Run all.** Training takes **~4–8 minutes** on free Colab CPU (it's NumPy, no GPU), so the rest of the notebook is quick.

## Cell 1 — Get the project

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yassine1158/WAKIL-AI.git"
if not os.path.exists("wakil_ai/transformer.py"):
    if not os.path.exists("WAKIL-AI"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("WAKIL-AI")
sys.path.insert(0, os.getcwd())

import inspect
import numpy as np
import matplotlib.pyplot as plt
from wakil_ai.tokenizer import CharTokenizer
from wakil_ai.text_data import load_text, split_ids, get_sequence_batch
from wakil_ai.language_model import BigramCounts, CharMLP, train_language_model
from wakil_ai.language_model import estimate_loss as estimate_mlp_loss
from wakil_ai import transformer
from wakil_ai.transformer import TinyGPT, train_transformer, estimate_loss

text = load_text()
tok = CharTokenizer.from_text(text)
train_ids, val_ids = split_ids(tok.encode(text))
print("Working directory:", os.getcwd())
print(f"{len(text):,} characters, vocabulary of {tok.vocab_size}")

## Why attention? The limit of Part 1

The Part 1 model glued 8 character vectors together and passed them through a layer. Two problems:
- **The memory is short and fixed.** For 100 characters, `W1` would need 100 × 16 rows, and each position would still have its own separate weights.
- **It can't choose.** To predict the letter after `"ROMEO: I love thee, Jul"`, the useful information is *where* "Jul" is, not a fixed slot.

**Self-attention** lets each position **look at all previous positions** and **decide how much each one matters**, using weights computed from the content itself.

### Query, Key, Value: a library analogy
Each token produces three vectors (with three learned matrices):
- **Query** $q$: *"what am I looking for?"*
- **Key** $k$: *"what do I contain?"* (like the label on a book)
- **Value** $v$: *"what information do I give if I'm chosen?"* (the content of the book)

The token compares its query with every key (dot product = similarity), turns the scores into weights with softmax, and takes the **weighted average of the values**:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + \text{mask}\right) V$$

- $\sqrt{d_k}$ keeps the scores in a reasonable range, so the softmax doesn't saturate.
- **Causal mask:** position $t$ may only look at positions $\le t$. Otherwise, during training, it would simply read the answer (the next character). We set future scores to $-\infty$, so their weight becomes 0.

## Cell 2 — Attention by hand on 4 tokens

**Expected output:**
- A 4×4 score matrix with `-inf` above the diagonal.
- Attention weights where each row sums to 1 and the first row is `[1, 0, 0, 0]`: the first token can only look at itself.

In [ ]:
rng = np.random.default_rng(0)
T, d = 4, 3                                   # 4 tokens, vectors of size 3
x = rng.standard_normal((T, d))               # the 4 token vectors (after embedding)
Wq, Wk, Wv = (rng.standard_normal((d, d)) for _ in range(3))   # learned in a real model

Q, K, V = x @ Wq, x @ Wk, x @ Wv
scores = Q @ K.T / np.sqrt(d)                              # (4, 4): similarity of every pair
mask = np.triu(np.ones((T, T), dtype=bool), k=1)           # True = future
scores[mask] = -np.inf
weights = np.exp(scores - scores.max(axis=1, keepdims=True))
weights /= weights.sum(axis=1, keepdims=True)              # softmax per row
out = weights @ V                                          # each output = weighted average of values

np.set_printoptions(precision=2, suppress=True)
print("Scores (query row i, key column j):\n", scores)
print("\nAttention weights (each row sums to 1):\n", weights)
print("\nRow sums:", weights.sum(axis=1))
print("\nOutput: one new vector per token, mixing information from the past:\n", out)

## Cell 3 — The real code: multi-head attention

The real layer does the same thing for a whole batch, with **several heads** in parallel. Our model has 4 heads of size 16 = 64. Each head can learn to look for something different (for example the previous letter, the start of the word, or the last space), and their outputs are concatenated and mixed by a final linear layer (`proj`).

The `backward()` uses the same rules you already know:
- matrix products: `dA = dOut @ B.T`, `dB = A.T @ dOut`
- the softmax derivative: `att * (datt - sum(datt * att))`
- masked positions have weight 0, so they receive zero gradient automatically.

In [ ]:
print(inspect.getsource(transformer.CausalSelfAttention))

## Cell 4 — The other pieces of a Transformer block

```
x ──► LayerNorm ──► Attention ──► (+) ──► LayerNorm ──► MLP ──► (+) ──►
  └────────────────────────────────┘  └──────────────────────────┘
          residual connection                residual connection
```

- **Positional embeddings:** attention is a weighted average, so it doesn't know the *order* of tokens ("dog bites man" and "man bites dog" would look the same). We add a learned vector per position: `x = tok_emb[ids] + pos_emb[0..T-1]`.
- **Residual connections** (`x = x + layer(x)`): each layer only adds a *correction*. In backprop, the gradient of `+` is 1, so it flows straight down through many layers without vanishing.
- **LayerNorm:** rescales each token vector to mean 0 and variance 1 (then applies learned scale and shift), which keeps training stable.
- **MLP** (64 → 256 → 64, ReLU): attention *gathers* information from other tokens; the MLP *processes* it, token by token.
- At the end: LayerNorm → Linear(64, 65) → **a prediction at every position at once**. A window of 32 characters gives 32 training examples in one forward pass.

In [ ]:
print(inspect.getsource(transformer.Block))
print(inspect.getsource(transformer.LayerNorm.forward))

## Cell 5 — Adam: a smarter gradient descent

Plain gradient descent (Phases 1–3) uses the same step size for every parameter. Transformers train much better with **Adam**, which keeps two running averages per parameter:
- $m$ = average gradient → **momentum**: keep going in a consistent direction, ignore noise
- $v$ = average **squared** gradient → parameters with big, noisy gradients get **smaller** steps, and rarely-updated ones get **bigger** steps

$$\theta \leftarrow \theta - \text{lr} \cdot \frac{\hat m}{\sqrt{\hat v} + \varepsilon}$$

(The "hat" versions correct the bias of the averages during the first steps, when they start at 0.)

In [ ]:
print(inspect.getsource(transformer.Adam.step))

## Cell 6 — Is the backpropagation correct? (tests)

The project's tests include a **gradient check of the entire Transformer** (every parameter of every layer, against numerical derivatives) and a test proving the **causal mask hides the future**: changing the last character doesn't change the predictions for earlier positions.

**Expected output:** `7 passed`.

In [ ]:
!python -m pytest -q tests/test_transformer.py

## Cell 7 — Train both models

First the Part 1 model (MLP, ~20 s) as a reference, then the Transformer (~4–8 min on Colab).

**Expected output for the Transformer** (our reference run, last decimals may differ):
```
step   500/5000 | ... | val loss ~2.0
...
step  5000/5000 | lr 0.0003 | train loss 1.5449 | val loss 1.7219
```
⚠️ The Transformer's loss is the average over **all 32 positions**, including the first ones, which see only 1 or 2 characters and can't do better than the bigram. So the comparison with the MLP (which always sees 8 characters) is slightly **unfair to the Transformer**, and it still wins.

In [ ]:
print("=== Part 1 model: MLP with embeddings ===")
mlp = CharMLP(tok.vocab_size)
train_language_model(mlp, train_ids, val_ids, steps=20000, eval_every=10000)
mlp_val = estimate_mlp_loss(mlp, val_ids)

print("\n=== Part 2 model: Transformer ===")
gpt = TinyGPT(tok.vocab_size, block_size=32, d_model=64, n_heads=4, n_layers=2)
print(f"{gpt.num_parameters():,} parameters | sees {gpt.context} characters")
history = train_transformer(gpt, train_ids, val_ids, steps=5000, batch_size=32, learning_rate=3e-3)
gpt_val = estimate_loss(gpt, val_ids)
bigram_val = BigramCounts(tok.vocab_size).fit(train_ids).loss(val_ids)

## Cell 8 — Results

In [ ]:
print(f"{'Model':<36}{'Parameters':>12}{'Context':>9}{'Val loss':>10}")
print(f"{'Uniform guess':<36}{0:>12,}{0:>9}{np.log(tok.vocab_size):>10.4f}")
print(f"{'Bigram (counting)':<36}{tok.vocab_size**2:>12,}{1:>9}{bigram_val:>10.4f}")
print(f"{'MLP + embeddings (Part 1)':<36}{sum(p.size for p in mlp.params.values()):>12,}{8:>9}{mlp_val:>10.4f}")
print(f"{'Transformer (Part 2)':<36}{gpt.num_parameters():>12,}{32:>9}{gpt_val:>10.4f}")

plt.figure(figsize=(9, 4))
plt.plot(history["step"], history["train_loss"], "o-", label="Transformer train")
plt.plot(history["step"], history["val_loss"], "o-", label="Transformer validation")
plt.axhline(mlp_val, color="green", linestyle="--", label="MLP (Part 1) validation")
plt.axhline(bigram_val, color="gray", linestyle="--", label="bigram")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Transformer vs. Part 1")
plt.show()

## Cell 9 — Side by side: what do they write?

**What to look for:** the Transformer usually keeps the structure of a dialogue better and produces more real words and plausible phrases. Both still invent words and don't make sense over a whole sentence. That's the honest level of ~100k parameters and a few minutes of CPU.

In [ ]:
for name, model in [("MLP (Part 1)", mlp), ("Transformer (Part 2)", gpt)]:
    sample = model.generate(tok.encode("ROMEO:\n"), 300, np.random.default_rng(1), temperature=0.8)
    print(f"================ {name} ================")
    print(tok.decode(sample), "\n")

## Cell 10 — Look inside the attention heads 🔍

For one sentence, we plot the attention weights of each head of the **last layer**: row = the position that is predicting, column = the position it looks at. Bright = high weight. The upper triangle is black: that's the causal mask.

**What to look for:** heads often specialize. In our reference run:
- some heads look mostly at the **last few characters**: a bright band along the diagonal, useful to finish a word;
- one head draws **vertical stripes**: many positions keep looking back at the same few characters near the start (around `ROMEO:` and the line break), a way to remember *who is speaking* and where the line began.

Patterns vary between training runs. You're looking at what *your* model learned, so describe what you see rather than expecting exactly this picture.

In [ ]:
sentence = "ROMEO:\nBut soft, what light thro"[: gpt.context]
gpt.forward(np.array([tok.encode(sentence)]))
att = gpt.blocks[-1].attn.cache[3][0]               # (heads, T, T) from the last forward pass
labels = [{"\n": "\\n", " ": "␣"}.get(c, c) for c in sentence]

fig, axes = plt.subplots(1, att.shape[0], figsize=(5 * att.shape[0], 5))
for h, ax in enumerate(axes):
    ax.imshow(att[h], cmap="viridis")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=7)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=7)
    ax.set_title(f"layer {len(gpt.blocks)}, head {h + 1}")
    ax.set_xlabel("looks at"); ax.set_ylabel("predicting after")
plt.tight_layout()
plt.show()

recent = [np.mean([att[h, t, max(0, t - 2):t + 1].sum() for t in range(len(labels))]) for h in range(att.shape[0])]
print("Share of attention on the last 3 characters (current + 2 previous), per head:", np.round(recent, 2))

## Cell 11 — Save and download

The file works with the project's scripts: `python generate.py --model wakil_gpt.npz --prompt "ROMEO:"`.

In [ ]:
MODEL_PATH = "wakil_gpt.npz"
gpt.save(MODEL_PATH, tok, {"name": "Wakil AI - TinyGPT", "val_loss": gpt_val, "pretrained_weights": False})
print(f"Saved {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1024:.0f} KB)")
try:
    from google.colab import files
    files.download(MODEL_PATH)
except ImportError:
    print("Not in Colab: the file is in", os.path.abspath(MODEL_PATH))

## Cell 12 — Phase 3 (Part 2) checklist

In [ ]:
test_run = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_transformer.py"], capture_output=True, text=True)
checks = [
    ("Attention by hand: rows sum to 1, first token only sees itself",
     np.allclose(weights.sum(axis=1), 1) and np.allclose(weights[0], [1, 0, 0, 0])),
    ("Transformer tests pass (gradient check + causal mask)", test_run.returncode == 0),
    ("Transformer beats the bigram", gpt_val < bigram_val),
    ("Transformer beats the Part 1 MLP", gpt_val < mlp_val),
    ("Validation loss < 1.80", gpt_val < 1.80),
    ("Model saved", os.path.exists(MODEL_PATH)),
]
for description, ok in checks:
    print(("✅ " if ok else "❌ ") + description)
print("\nPHASE 3 COMPLETE 🎉" if all(ok for _, ok in checks) else "\nNot complete yet.")

## Wakil AI vs. real large language models: an honest comparison

| Model | Parameters | Training data |
|---|---|---|
| **Wakil AI TinyGPT** | **~110 thousand** | 1 million characters, a few CPU minutes |
| GPT-2 small (2019) | 124 million | ~40 GB of web text |
| GPT-3 (2020) | 175 billion | ~300 billion tokens, thousands of GPUs |
| Claude, GPT-4 and similar | not published (much larger) | trillions of tokens, huge GPU clusters, large teams |

The **architecture is the same family**: embeddings, positional information, attention + MLP blocks with residuals and normalization, next-token prediction, cross-entropy, backpropagation, Adam. The difference is **scale** (parameters, data, compute) plus a lot of engineering. Assistants like Claude also go through further training (for example on human feedback) to become helpful and safe conversational assistants, which goes far beyond next-token prediction on raw text.

## Experiments
1. `n_layers=4` or `d_model=128`: better loss? How much slower?
2. `n_heads=1` vs `n_heads=8` (same `d_model`): does the number of heads matter here?
3. `block_size=64`: does a longer memory help?
4. Remove the positional embeddings (set `gpt.params["pos_emb"][:] = 0` before training and skip its update). What happens?

**Next: Phase 4** — an application where you type a prompt or draw a digit and Wakil AI answers.